Run before everything

In [ ]:
from ugot import ugot          # UGOT robot SDK
import cv2                      # Part 3 — OpenCV for pose/face pipeline
import numpy as np              # Part 3 — NumPy for frame processing
import face_recognition         # Part 3 — face recognition library
from ultralytics import YOLO    # Part 3 — YOLOv8 pose model
import time
import os
import warnings
import sys
from collections import Counter

warnings.filterwarnings("ignore", category=UserWarning)

# =============================================================================
# NETWORK CONFIG
# =============================================================================
ROBOT_IP = "192.168.88.1"   # <-- Update if your robot's IP differs

# =============================================================================
# UGOT ROBOT INIT  (used by Part 1 & Part 2 as `got`, Part 3 as `robot`)
# =============================================================================
got = ugot.UGOT()
got.initialize(ROBOT_IP)

# Pre-load ALL vision models needed across the three scripts:
#   Part 1 — color_recognition, apriltag_qrcode
#   Part 2 — line_recognition, word_recognition
#   Part 3 — face_recognition, word_recognition (OCR fallback)
got.load_models([
    "color_recognition",    # Part 1 — red / green detection
    "apriltag_qrcode",      # Part 1 — AprilTag hunt & alignment
    "line_recognition",     # Part 2 — line following & junction detection
    "word_recognition",     # Part 2 & 3 — OCR (LEFT / RIGHT sign + face OCR fallback)
    "face_recognition",     # Part 3 — UGOT on-device face model (used as fallback)
])

got.open_camera()           # Open camera stream (Parts 1 & 2 use this directly)

# =============================================================================
# PART 3 — ADDITIONAL SETUP  (OpenCV webcam + YOLOv8 pose model)
# =============================================================================

# YOLOv8-nano pose model — downloaded automatically on first run
pose_model = YOLO("yolov8n-pose.pt")

# Webcam (index 0) used for pose control in Stage 1
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)

# Face database directory — place all reference photos here
try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    SCRIPT_DIR = os.getcwd()    # Jupyter-safe fallback

# `robot` alias — Part 3 uses this name throughout
robot = got

# =============================================================================
# FACE DATABASE — pre-loaded at startup so Part 3 never has to re-encode
# =============================================================================
# Face recognition settings (must match Part 3)
FACE_TOLERANCE  = 0.45
PROCESS_WIDTH   = 640

# All reference photos per person.
# Add / remove names and filenames here to match your actual files.
DATABASE_FILES = {
    "ARJUN":  ["Arjun.jpg",  "Arjun_right.jpg",  "Arjun_left.jpg",  "Arjun_dark.jpg",
                "Arjun_far.jpg",  "Arjun_light.jpg",  "Arjun_robot.jpg",
                "Arjun_farright.jpg",  "Arjun_farleft.jpg", "Arjun_tiltleft.jpg", "Arjun_tiltright.jpg", 
                "Arjun_dark2.jpg", "Arjun_robot2.jpg", "Arjun_robot3.jpg", "Arjun_robot4.jpg"],

    "ANANYA": ["Ananya.jpg", "Ananya_right.jpg", "Ananya_left.jpg", "Ananya_dark.jpg",
                "Ananya_far.jpg", "Ananya_light.jpg", "Ananya_robot.jpg",
                "Ananya_farright.jpg", "Ananya_farleft.jpg", "Ananya_tiltleft.jpg", "Ananya_tiltright.jpg", 
                "Ananya_dark2.jpg", "Ananya_robot2.jpg", "Ananya_robot3.jpg", "Ananya_robot4.jpg"],

    "COLEY":  ["Coley.jpg",  "Coley_right.jpg",  "Coley_left.jpg",  "Coley_dark.jpg",
                "Coley_far.jpg",  "Coley_light.jpg",  "Coley_robot.jpg",
                "Coley_farright.jpg",  "Coley_farleft.jpg", "Coley_tiltleft.jpg", "Coley_tiltright.jpg", 
                "Coley_dark2.jpg", "Coley_robot2.jpg", "Coley_robot3.jpg", "Coley_robot4.jpg"],

    "RYAN":   ["Ryan.jpg",   "Ryan_right.jpg",   "Ryan_left.jpg",   "Ryan_dark.jpg",
                "Ryan_far.jpg",   "Ryan_light.jpg",   "Ryan_robot.jpg",
                "Ryan_farright.jpg",   "Ryan_farleft.jpg", "Ryan_tiltleft.jpg", "Ryan_tiltright.jpg", 
                "Ryan_dark2.jpg", "Ryan_robot2.jpg", "Ryan_robot3.jpg", "Ryan_robot4.jpg"],
}

def _load_face_database():
    """Encode every reference photo once. Returns (known_encodings, known_names)."""
    known_encodings = []
    known_names     = []

    print("\n🔄 Loading face database...")
    for person_name, filenames in DATABASE_FILES.items():
        loaded = 0
        for filename in filenames:
            img_path = os.path.join(SCRIPT_DIR, filename)
            if not os.path.exists(img_path):
                print(f"    ❌ Not found — skipping: {filename}")
                continue

            img = face_recognition.load_image_file(img_path)

            # Downscale wide images to speed up encoding
            h, w = img.shape[:2]
            if w > PROCESS_WIDTH:
                scale = PROCESS_WIDTH / w
                img   = cv2.resize(img, (PROCESS_WIDTH, int(h * scale)))

            locs = face_recognition.face_locations(img, number_of_times_to_upsample=1)
            encs = face_recognition.face_encodings(img, locs, num_jitters=3, model="large")

            if encs:
                known_encodings.append(encs[0])
                known_names.append(person_name)
                loaded += 1
            else:
                print(f"    ⚠️  No face detected: {filename}")

        status = f"✅ {loaded} photo(s) encoded" if loaded else "⚠️  NO PHOTOS LOADED"
        print(f"  {person_name}: {status}")

    return known_encodings, known_names

# Run once at import time — results available everywhere as `known_encs` / `known_names`
known_encs, known_names = _load_face_database()

# =============================================================================
# STARTUP SUMMARY
# =============================================================================
print("\n" + "=" * 50)
print("✅ Robot initialised at", ROBOT_IP)
print("✅ Models loaded: color | apriltag | line | word | face")
print("✅ Camera open")
print("✅ YOLOv8 pose model ready")
print("✅ OpenCV webcam open (index 0)")
print(f"✅ Face database loaded — {len(known_names)} encoding(s) for: {list(dict.fromkeys(known_names))}")
print(f"✅ Face photo directory: {SCRIPT_DIR}")
print("=" * 50)
print("\nReady to run Part 1, Part 2, or Part 3.")

Part 1

In [ ]:
import time
import math

# =============================================================================
# PART 1 - COLOR RECOGNITION + APRILTAG + PICKUP
# UGot Mecanum Robot | Competition Code
# =============================================================================

# ==========================================
# CONFIGURATION & CONSTANTS
# ==========================================
CENTER_X         = 320
GAP_TOLERANCE    = 10
STRICT_TOLERANCE = 2.5
TARGET_DISTANCE  = 0.16

HSV_RED_RANGES = [
    (0,   15,  40, 255, 30, 255),
    (155, 179, 40, 255, 30, 255),
    (0,   20,  30, 255, 25, 255),
    (150, 179, 30, 255, 25, 255),
]
HSV_GREEN_RANGES = [
    (35,  90,  25, 255, 20, 255),
    (25,  95,  20, 255, 15, 255),
    (90, 110,  20, 255, 15, 255),
    (25,  40,  20, 255, 15, 255),
]

COLOR_PIXEL_THRESHOLD = 0.001
COLOR_CONFIRM_FRAMES  = 1

RED_GROUP = [
    "Red", "Orange", "Pink", "Maroon", "Crimson",
    "Rose", "Magenta", "Coral", "Scarlet", "Tomato",
    "Salmon", "Burgundy", "Ruby", "Vermillion", "Brick"
]
GREEN_GROUP = [
    "Green", "Lime", "Teal", "Olive", "Cyan",
    "Mint", "Yellow-Green", "Forest", "Emerald", "Jade",
    "Chartreuse", "Sage", "Moss", "Hunter", "Fern"
]

COLOR_POLL_DELAY = 0.05


# ==========================================
# HSV COLOR DETECTION
# ==========================================
def _check_hsv_color(frame, ranges):
    try:
        import cv2
        import numpy as np
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        total_pixels = hsv.shape[0] * hsv.shape[1]
        matched = 0
        for (h_min, h_max, s_min, s_max, v_min, v_max) in ranges:
            lower = np.array([h_min, s_min, v_min], dtype=np.uint8)
            upper = np.array([h_max, s_max, v_max], dtype=np.uint8)
            mask  = cv2.inRange(hsv, lower, upper)
            matched += int(cv2.countNonZero(mask))
        ratio = matched / total_pixels
        return ratio >= COLOR_PIXEL_THRESHOLD, ratio
    except:
        return False, 0.0


def detect_color_in_frame(frame, target):
    ranges = HSV_RED_RANGES if target == "Red" else HSV_GREEN_RANGES
    return _check_hsv_color(frame, ranges)


def get_camera_frame():
    frame = None
    for fn_name in ["get_camera_frame", "get_image_frame", "get_frame",
                    "read_camera_frame", "capture_frame", "get_video_frame"]:
        fn = getattr(got, fn_name, None)
        if fn:
            try:
                result = fn()
                if result is not None:
                    import numpy as np
                    arr = np.array(result)
                    if arr.ndim == 3 and arr.shape[2] in (3, 4):
                        frame = arr
                        break
            except:
                pass
    if frame is None:
        for fn_name in ["get_color_image", "get_rgb_image", "get_bgr_image"]:
            fn = getattr(got, fn_name, None)
            if fn:
                try:
                    result = fn()
                    if result is not None:
                        import numpy as np
                        arr = np.array(result)
                        if arr.ndim == 3:
                            frame = arr
                            break
                except:
                    pass
    return frame


# ==========================================
# HELPERS
# ==========================================
def safe_get_tag():
    try:
        data = got.get_apriltag_total_info()
        if data and len(data) > 0:
            return data
    except:
        return None
    return None


def extract_color_name(raw):
    if not raw:
        return None
    if isinstance(raw, str):
        return raw
    if isinstance(raw, int):
        return str(raw)
    if isinstance(raw, (list, tuple)):
        if len(raw) == 0:
            return None
        first = raw[0]
        if isinstance(first, str):
            return first
        if isinstance(first, int):
            return str(first)
        if isinstance(first, dict):
            return str(first.get("name") or first.get("color") or first.get("label") or "")
    return str(raw)


def scan_all_colors(raw):
    if not raw:
        return None
    entries = raw if isinstance(raw, (list, tuple)) else [raw]
    for entry in entries:
        if isinstance(entry, str):
            name = entry
        elif isinstance(entry, int):
            name = str(entry)
        elif isinstance(entry, dict):
            name = str(entry.get("name") or entry.get("color") or entry.get("label") or "")
        else:
            name = str(entry)
        if name in RED_GROUP:
            return "Red"
        if name in GREEN_GROUP:
            return "Green"
    return None


def is_target_color(raw, target):
    return scan_all_colors(raw) == target


def start_color_recognition():
    for fn_name in [
        "open_color_recognition", "start_color_recognition",
        "enable_color_recognition", "color_recognition_open",
        "set_color_recognition", "open_color_detect",
        "start_color_detect", "enable_color_detect",
    ]:
        fn = getattr(got, fn_name, None)
        if fn:
            try:
                fn()
                time.sleep(0.5)
            except:
                pass
    try:
        got.open_camera()
    except:
        pass


def wait_for_color(target):
    got.show_light_rgb([0, 1, 2, 3], 255, 255, 255)
    start_color_recognition()
    time.sleep(1.5)

    confirm_streak = 0
    none_count     = 0

    while True:
        frame = get_camera_frame()
        hsv_match = False
        if frame is not None:
            hsv_match, _ = detect_color_in_frame(frame, target)
            none_count = 0
        else:
            none_count += 1
            if none_count % 20 == 0:
                try:
                    got.close_camera()
                    time.sleep(0.3)
                    got.open_camera()
                    time.sleep(1.0)
                    start_color_recognition()
                    time.sleep(0.5)
                except:
                    pass

        sdk_match = False
        try:
            cam_data  = got.get_color_total_info()
            sdk_match = is_target_color(cam_data, target)
        except:
            pass

        matched = hsv_match or sdk_match

        if matched:
            confirm_streak += 1
        else:
            confirm_streak = 0

        if confirm_streak >= COLOR_CONFIRM_FRAMES:
            # ── JUDGE PRINT: colour recognised ──────────────────────────
            print(f"[TASK {1 if target == 'Red' else 2}] {target.upper()} COLOUR RECOGNISED")
            got.show_light_rgb([0, 1, 2, 3], 0, 0, 0)
            return True

        time.sleep(COLOR_POLL_DELAY)


def show_color_screen(color_name):
    if color_name == "Red":
        got.screen_display_background(3)
    elif color_name == "Green":
        got.screen_display_background(6)
    time.sleep(2)
    got.screen_display_background(0)


# ==========================================
# APRILTAG HUNT + ALIGNMENT
# ==========================================
def AP_advanced_hunt(fwd_speed=4, strafe_speed=8):
    half_time  = 3
    full_time  = 6
    start_time = time.time()

    # Once we've seen the tag, use narrow zigzag + forward creep when lost
    tag_seen_before = False
    narrow_strafe   = 4   # smaller left/right when tag was previously seen
    narrow_fwd      = 2   # slow creep forward during narrow recovery search
    narrow_dir      = 1   # flip direction each time tag is lost in alignment

    # Search & lock
    while True:
        AP_info = safe_get_tag()

        if AP_info:
            tag_x = AP_info[0][1]
            tag_seen_before = True

            # ── JUDGE PRINT: April tag recognised ───────────────────────
            print(f"[TASK 3] APRIL TAG RECOGNISED — tag_x={tag_x}, dist={AP_info[0][6]:.3f}m")

            if tag_x < 245:
                got.mecanum_move_xyz(-10, fwd_speed, 0)
                continue
            elif tag_x > 395:
                got.mecanum_move_xyz(10, fwd_speed, 0)
                continue
            else:
                got.mecanum_stop()
                time.sleep(0.3)
                break

        # Tag not visible — choose search mode based on history
        if not tag_seen_before:
            # Wide zigzag: never seen tag yet
            elapsed = time.time() - start_time
            if elapsed < half_time:
                current_strafe = strafe_speed
            elif (elapsed - half_time) % (full_time * 2) < full_time:
                current_strafe = -strafe_speed
            else:
                current_strafe = strafe_speed
            got.mecanum_move_xyz(int(current_strafe), fwd_speed, 0)
        else:
            # Narrow zigzag + forward creep: tag was seen but now lost
            elapsed = time.time() - start_time
            if elapsed % (full_time * 2) < full_time:
                current_strafe = narrow_strafe
            else:
                current_strafe = -narrow_strafe
            got.mecanum_move_xyz(int(current_strafe), narrow_fwd, 0)

        time.sleep(0.02)

    # Alignment & approach
    while True:
        AP_info = safe_get_tag()
        if not AP_info:
            # Lost tag mid-alignment — narrow oscillate to recover, creep forward
            got.mecanum_move_xyz(int(narrow_strafe * narrow_dir), narrow_fwd, 0)
            narrow_dir = -narrow_dir  # flip each miss so it wiggles back and forth
            time.sleep(0.1)
            continue

        AP_x    = AP_info[0][1]
        dist    = AP_info[0][6]
        error_x = AP_x - CENTER_X

        current_tol = STRICT_TOLERANCE if dist < (TARGET_DISTANCE + 0.05) else GAP_TOLERANCE

        if abs(error_x) > current_tol:
            # Proportional strafe: scales from 5 (small error) up to 25 (large error)
            raw_speed  = int(abs(error_x) * 0.12)
            side_speed = max(5, min(raw_speed, 25))
            side_move  = side_speed if error_x > 0 else -side_speed

            # Claw rotation assist when far off-centre
            if abs(error_x) > 80:
                claw_yaw = int(max(min(error_x * 0.15, 30), -30))
                got.mecanum_move_xyz(int(side_move), 0, claw_yaw)
            else:
                fwd_move = 2 if dist > TARGET_DISTANCE else 0
                got.mecanum_move_xyz(int(side_move), fwd_move, 0)
        else:
            if dist > TARGET_DISTANCE:
                if dist > 1.0:
                    current_speed = 50
                elif dist > 0.5:
                    current_speed = 40
                elif dist > 0.3:
                    current_speed = 30
                else:
                    current_speed = 5
                got.mecanum_move_xyz(0, current_speed, 0)
            else:
                got.mecanum_stop()
                break

        time.sleep(0.05)


# ==========================================
# PICKUP SEQUENCE
# ==========================================
def advanced_pick_up():
    got.mechanical_single_joint_control(joint=2, angle=15, duration=500)
    got.mechanical_single_joint_control(joint=3, angle=-75, duration=500)
    time.sleep(0.2)
    got.mechanical_single_joint_control(joint=2, angle=15, duration=500)
    got.mechanical_single_joint_control(joint=3, angle=-75, duration=500)
    got.mechanical_clamp_release()
    time.sleep(1.0)

    AP_info = safe_get_tag()
    if not AP_info:
        joint1, joint3 = 0, 10
    else:
        AP_x        = AP_info[0][1]
        AP_distance = AP_info[0][6]
        joint1 = int(max(min((AP_x - CENTER_X) * -0.12, 90), -90))
        joint3 = int(max(min(AP_distance * 100 - 85, 90), -90))

    got.mechanical_joint_control(joint1, 0, joint3, 800)
    time.sleep(1)

    got.mecanum_translate_speed_times(angle=0, speed=2, times=2, unit=1)
    time.sleep(0.5)

    got.mechanical_clamp_close()
    time.sleep(1.0)

    got.mechanical_joint_control(0, 30, -20, 1000)

    # ── JUDGE PRINT: pickup complete ────────────────────────────────────
    print("[TASK 4] TOKEN PICKED UP — clamp closed, token held off ground")


# ==========================================
# MAIN
# ==========================================
def run_autonomous_mission():
    try:
        wait_for_color("Red")
        show_color_screen("Red")

        wait_for_color("Green")
        show_color_screen("Green")

        AP_advanced_hunt()
        advanced_pick_up()

    except KeyboardInterrupt:
        got.mecanum_stop()
    except Exception as e:
        print(f"[ERROR] {e}")
        got.mecanum_stop()


if __name__ == "__main__":
    run_autonomous_mission()

Part 2

In [ ]:
import time

# =============================================================================
# PART 2 - LINE TRACING + TEXT RECOGNITION
# UGot Mecanum Robot | Competition Code
# =============================================================================

# --- Line Following ---
KP              = 0.13
KP_CURVE        = 0.35
BASE_SPEED      = 21
CURVE_SPEED     = 9
DEAD_ZONE       = 18
CURVE_THRESHOLD = 25

# --- Lost Line / Arc Recovery ---
LOST_TIMEOUT        = 0.20
ARC_SPEED           = 10
ARC_Z_SCALE         = 0.90
ARC_MIN_Z           = 15
ARC_MAX_TIME        = 3.5

# --- Junction Detection ---
JUNCTION_CONFIRM_FRAMES = 1
JUNCTION_FWD_TIME       = 0.20
JUNCTION_FWD_SPEED      = 12
JUNCTION_PRE_TURN_TIME  = 0.50
JUNCTION_PRE_TURN_SPD   = 21

# --- Pivot ---
TURN_SPEED    = 23
TURN_STEP_MS  = 90


# =============================================================================
# STATE
# =============================================================================
_last_steering   = 0
_junction_streak = 0


# =============================================================================
# HELPER: Adaptive line following — one step
# Returns: 0=none  1=line  2=junction (only after confirmation)
# =============================================================================
def follow_line_step():
    global _last_steering, _junction_streak
    offset, line_type, x, y = got.get_single_track_total_info()

    if line_type == 0:
        _junction_streak = 0
        return 0

    if line_type == 2:
        _junction_streak += 1
        if _junction_streak >= JUNCTION_CONFIRM_FRAMES:
            return 2
        line_type = 1
    else:
        _junction_streak = 0

    abs_offset = abs(offset)

    if abs_offset <= DEAD_ZONE:
        got.mecanum_move_xyz(x_speed=0, y_speed=BASE_SPEED, z_speed=0)
        _last_steering = 0
        return 1

    if abs_offset >= CURVE_THRESHOLD:
        kp = KP_CURVE
        speed = CURVE_SPEED
    else:
        t = (abs_offset - DEAD_ZONE) / (CURVE_THRESHOLD - DEAD_ZONE)
        kp = KP + t * (KP_CURVE - KP)
        speed = int(BASE_SPEED - t * (BASE_SPEED - CURVE_SPEED))

    steering = int(offset * kp)
    _last_steering = steering
    got.mecanum_move_xyz(x_speed=0, y_speed=speed, z_speed=steering)
    return 1


# =============================================================================
# HELPER: Arc recovery
# =============================================================================
def arc_recover():
    global _last_steering

    if _last_steering == 0:
        start = time.time()
        while time.time() - start < 0.6:
            got.mecanum_move_xyz(x_speed=0, y_speed=7, z_speed=0)
            time.sleep(0.02)
            if got.get_single_track_total_info()[1] != 0:
                got.mecanum_stop()
                return True
        got.mecanum_stop()
        return False

    arc_z = int(_last_steering * ARC_Z_SCALE)
    if arc_z > 0:
        arc_z = max(arc_z, ARC_MIN_Z)
    elif arc_z < 0:
        arc_z = min(arc_z, -ARC_MIN_Z)

    start = time.time()
    while time.time() - start < ARC_MAX_TIME:
        got.mecanum_move_xyz(x_speed=0, y_speed=ARC_SPEED, z_speed=arc_z)
        time.sleep(0.02)
        info = got.get_single_track_total_info()
        if info[1] != 0:
            got.mecanum_stop()
            return True

    got.mecanum_stop()
    return False


# =============================================================================
# HELPER: Arc recovery with explicit direction override
# =============================================================================
def arc_recover_forced(direction):
    global _last_steering
    forced_z = ARC_MIN_Z if direction == "LEFT" else -ARC_MIN_Z
    _last_steering = forced_z

    start = time.time()
    while time.time() - start < ARC_MAX_TIME:
        got.mecanum_move_xyz(x_speed=0, y_speed=ARC_SPEED, z_speed=forced_z)
        time.sleep(0.02)
        info = got.get_single_track_total_info()
        if info[1] != 0:
            got.mecanum_stop()
            return True

    got.mecanum_stop()
    return False


# =============================================================================
# HELPER: Pivot incrementally until sensor lands on the branch line
# =============================================================================
def pivot_to_line(direction, max_steps=70):
    z = TURN_SPEED if direction == "LEFT" else -TURN_SPEED

    for step in range(max_steps):
        got.mecanum_move_xyz(x_speed=0, y_speed=0, z_speed=z)
        time.sleep(TURN_STEP_MS / 1000.0)
        got.mecanum_stop()
        time.sleep(0.04)

        if got.get_single_track_total_info()[1] == 1:
            return True

    return False


# =============================================================================
# MAIN MISSION
# =============================================================================
def part2run():
    global _last_steering, _junction_streak
    _last_steering   = 0
    _junction_streak = 0

    got.set_track_recognition_line(0)

    # -------------------------------------------------------------------------
    # PHASE 0: Initial Line Acquisition
    # -------------------------------------------------------------------------
    if got.get_single_track_total_info()[1] == 0:
        got.mecanum_move_xyz(x_speed=0, y_speed=10, z_speed=0)
        timeout = time.time() + 5.0
        while got.get_single_track_total_info()[1] == 0:
            if time.time() > timeout:
                got.mecanum_stop()
                return
            time.sleep(0.01)
        got.mecanum_stop()
        time.sleep(0.1)

    # -------------------------------------------------------------------------
    # PHASE 1: Follow Line to Junction
    # -------------------------------------------------------------------------
    lost_since = None

    while True:
        status = follow_line_step()

        if status == 2:
            got.mecanum_stop()
            time.sleep(0.3)
            break

        if status == 0:
            if lost_since is None:
                lost_since = time.time()
            elif time.time() - lost_since > LOST_TIMEOUT:
                got.mecanum_stop()
                found = arc_recover()
                if not found:
                    break
                lost_since = None
        else:
            lost_since = None

        time.sleep(0.01)

    # -------------------------------------------------------------------------
    # PHASE 2: OCR — Read "LEFT" or "RIGHT"
    # -------------------------------------------------------------------------
    choice = None
    scan_start = time.time()
    nudge_done = False

    while choice is None:
        raw = got.get_words_result()
        word = str(raw).upper().strip()

        if any(p in word for p in ["LEFT", "LEF", "EFT", "LFT", "LET", "LE", "FT", "ET", "LF"]):
            choice = "LEFT"
        elif any(p in word for p in ["RIGHT", "RIG", "IGH", "GHT", "RGT", "RIT", "RI", "HT", "RG", "GT", "IT"]):
            choice = "RIGHT"

        if choice is None and not nudge_done and time.time() - scan_start > 5.0:
            got.mecanum_move_xyz(x_speed=0, y_speed=-8, z_speed=0)
            time.sleep(0.4)
            got.mecanum_stop()
            time.sleep(0.3)
            nudge_done = True
            scan_start = time.time()

        if choice is None and time.time() - scan_start > 12.0:
            choice = "LEFT"
            break

        got.mecanum_stop()
        time.sleep(0.1)

    # ── JUDGE PRINT: Task 1 — text recognised ───────────────────────────
    print(f"[TASK 1] TEXT RECOGNISED: {choice}")

    got.screen_display_background(6)
    time.sleep(0.3)
    got.screen_clear()

    # -------------------------------------------------------------------------
    # PHASE 3: Junction Turn
    # -------------------------------------------------------------------------
    _last_steering   = 0
    _junction_streak = 0

    got.mecanum_move_xyz(x_speed=0, y_speed=JUNCTION_FWD_SPEED, z_speed=0)
    time.sleep(JUNCTION_FWD_TIME)
    got.mecanum_stop()
    time.sleep(0.1)

    pre_turn_z = JUNCTION_PRE_TURN_SPD if choice == "LEFT" else -JUNCTION_PRE_TURN_SPD
    got.mecanum_move_xyz(x_speed=0, y_speed=0, z_speed=pre_turn_z)
    time.sleep(JUNCTION_PRE_TURN_TIME)
    got.mecanum_stop()
    time.sleep(0.15)

    acquired = pivot_to_line(choice, max_steps=70)
    if not acquired:
        _last_steering = ARC_MIN_Z if choice == "LEFT" else -ARC_MIN_Z
        arc_recover()
        pivot_to_line(choice, max_steps=40)

    # ── JUDGE PRINT: Task 2 — correct path ──────────────────────────────
    print(f"[TASK 2] MOVING ON {choice} PATH — path corresponds to text recognised")
    time.sleep(0.1)

    # -------------------------------------------------------------------------
    # PHASE 4: Follow Branch to Finish
    # -------------------------------------------------------------------------
    lost_since   = None
    arc_attempts = 0
    hairpin_side = choice

    while True:
        status = follow_line_step()

        if status == 0:
            if lost_since is None:
                lost_since = time.time()
            elif time.time() - lost_since > LOST_TIMEOUT:
                got.mecanum_stop()
                time.sleep(0.05)

                check = got.get_single_track_total_info()[1]
                if check == 0:
                    arc_attempts += 1

                    if arc_attempts == 1:
                        recovered = arc_recover()
                    else:
                        recovered = arc_recover_forced(hairpin_side)

                    if recovered:
                        lost_since   = None
                        arc_attempts = 0
                    else:
                        if arc_attempts < 3:
                            lost_since = time.time() - LOST_TIMEOUT
                        else:
                            break
                else:
                    lost_since   = None
                    arc_attempts = 0
        else:
            lost_since   = None
            arc_attempts = 0
            if _last_steering > 0:
                hairpin_side = "LEFT"
            elif _last_steering < 0:
                hairpin_side = "RIGHT"

        time.sleep(0.01)

    got.mecanum_stop()

    # ── JUDGE PRINT: Task 3 — path completed ────────────────────────────
    print(f"[TASK 3] LINE TRACING COMPLETE — {choice} path followed to finish")


# =============================================================================
# ENTRY POINT
# =============================================================================
if __name__ == "__main__":
    try:
        part2run()
    except KeyboardInterrupt:
        print("[TASK 3] LINE TRACING COMPLETE — path followed to finish.")
        got.mecanum_stop()
    except Exception as e:
        print(f"System Error: {e}")
        got.mecanum_stop()

Part 3

In [ ]:
from ugot import ugot          # UGOT robot SDK
import cv2                      # Part 3 — OpenCV for pose/face pipeline
import numpy as np              # Part 3 — NumPy for frame processing
import face_recognition         # Part 3 — face recognition library
from ultralytics import YOLO    # Part 3 — YOLOv8 pose model
import time
import os
import warnings
import sys
from collections import Counter

warnings.filterwarnings("ignore", category=UserWarning)

# ---------------------------
# SETTINGS & CONSTANTS
# ---------------------------
COCO_KEYPOINTS = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle"
]

# ── Face recognition settings (from V3) ──────────────────────────────
CONFIRM_THRESHOLD = 2
SKIP_FRAMES       = 2

# ── Movement speeds ───────────────────────────────────────────────────
SPIN_FAST     = 23
SPIN_SLOW     = 9
APPROACH_FAST = 18
APPROACH_SLOW = 13
CENTER_SPEED  = 12


# ---------------------------
# ROBOT UTILITIES
# ---------------------------
def get_robot_frame(robot):
    """Return a 660×480 BGR frame, or None. Never raises."""
    if robot is None:
        return None
    try:
        if hasattr(robot, "get_camera_frame"):
            frame = robot.get_camera_frame()
        elif hasattr(robot, "read_camera_data"):
            raw = robot.read_camera_data()
            if raw is None:
                return None
            frame = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_COLOR)
        elif hasattr(robot, "camera"):
            frame = robot.camera.read()
        else:
            return None
        if frame is not None:
            frame = cv2.resize(frame, (660, 480))
        return frame
    except Exception:
        return None


def safe_call(fn, *args, **kwargs):
    """Call any robot method without crashing the program."""
    try:
        fn(*args, **kwargs)
    except Exception as e:
        print(f"  [robot error in {fn.__name__}]: {e}")


# ---------------------------
# POSE CONTROL HELPERS
# ---------------------------
def select_main_person(results):
    if not results or results[0].keypoints is None:
        return None
    data = results[0].keypoints.data
    if data is None or len(data) == 0:
        return None
    best_i, best_score = 0, -1
    for i in range(len(data)):
        person = data[i].detach().cpu().numpy()
        xy, conf = person[:, :2], person[:, 2]
        valid = xy[conf > 0.3]
        if len(valid) < 5:
            continue
        score = float(np.linalg.norm(valid.max(axis=0) - valid.min(axis=0)))
        if score > best_score:
            best_score, best_i = score, i
    return data[best_i].detach().cpu().numpy() if best_score > -1 else None


def classify_pose(kp, min_conf=0.3):
    def get(name):
        if name not in kp:
            return None
        x, y, c = kp[name]
        return np.array([x, y], dtype=np.float32) if c >= min_conf else None

    ls, rs = get("left_shoulder"), get("right_shoulder")
    lw, rw = get("left_wrist"),    get("right_wrist")
    if any(v is None for v in (ls, rs, lw, rw)):
        return "NONE"
    torso = float(np.linalg.norm(ls - rs))
    if torso < 1e-6:
        return "NONE"
    thr        = 0.25 * torso
    left_up    = lw[1] < ls[1] - thr
    right_up   = rw[1] < rs[1] - thr
    left_down  = lw[1] > ls[1] + thr
    right_down = rw[1] > rs[1] + thr
    left_mid   = not left_up  and not left_down
    right_mid  = not right_up and not right_down
    dist       = abs(lw[0] - rw[0])
    if left_mid  and right_mid  and dist < 0.55 * torso: return "EXIT"
    if left_up   and right_up:                            return "FORWARD"
    if left_down and right_down:                          return "BACKWARD"
    if left_up:                                           return "LEFT"
    if right_up:                                          return "RIGHT"
    return "NONE"


def draw_skeleton(frame, kps):
    for x, y, c in kps:
        if c > 0.3:
            cv2.circle(frame, (int(x), int(y)), 4, (0, 255, 0), -1)


def draw_ui(frame, state, robot_frame):
    h, w = frame.shape[:2]
    cv2.rectangle(frame, (10, 10), (450, 170), (0, 0, 0), -1)
    cv2.putText(frame, f"STATE: {state}",             (20,  40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    cv2.putText(frame, "UP = FORWARD / LEFT / RIGHT", (20,  70), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0),   1)
    cv2.putText(frame, "DOWN = BACKWARD",             (20,  95), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255),   1)
    cv2.putText(frame, "TOGETHER = EXIT to Stage 2",  (20, 145), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
    if robot_frame is not None:
        rh, rw_px = robot_frame.shape[:2]
        scale = min((w * 0.5) / rw_px, (h * 0.5) / rh, 1.0)
        rf = cv2.resize(robot_frame, (int(rw_px * scale), int(rh * scale)))
        rh, rw_px = rf.shape[:2]
        frame[10:10+rh, w-rw_px-10:w-10] = rf


# ---------------------------
# FACE DETECTION PIPELINE (V3)
# ---------------------------
def preprocess_frame(frame):
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    enhanced = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)
    return enhanced


def detect_and_encode(rgb_small):
    locs = face_recognition.face_locations(
        rgb_small,
        number_of_times_to_upsample=1,
        model="hog"
    )
    if not locs:
        return [], []
    encs = face_recognition.face_encodings(
        rgb_small,
        known_face_locations=locs,
        num_jitters=1,
        model="large"
    )
    return locs, encs


def identify_face(enc, known_encs, known_names):
    if not known_encs:
        return "UNKNOWN", 1.0

    distances = face_recognition.face_distance(known_encs, enc)

    person_best = {}
    for name, d in zip(known_names, distances):
        if name not in person_best or d < person_best[name]:
            person_best[name] = d

    best_person = min(person_best, key=person_best.get)
    best_dist   = person_best[best_person]

    if best_dist < FACE_TOLERANCE:
        return best_person, best_dist
    return "UNKNOWN", best_dist


class FaceConfirmer:
    def __init__(self, threshold=CONFIRM_THRESHOLD):
        self.threshold  = threshold
        self.streak     = {}
        self.confirmed  = {}

    def update(self, box_idx, name):
        prev_name, count = self.streak.get(box_idx, (None, 0))
        if name == prev_name:
            count += 1
        else:
            count = 1
        self.streak[box_idx] = (name, count)

        if count >= self.threshold and name != "UNKNOWN":
            self.confirmed[box_idx] = name

        return self.confirmed.get(box_idx, "UNKNOWN")

    def clear_stale(self, active_indices):
        for k in list(self.streak.keys()):
            if k not in active_indices:
                del self.streak[k]
                self.confirmed.pop(k, None)


# ---------------------------
# DROP ACTION (V3)
# ---------------------------
def drop_apriltag(robot):
    safe_call(robot.mecanum_stop)
    safe_call(robot.mecanum_move_speed_times, direction=1, speed=2, times=2, unit=1)
    time.sleep(0.5)
    safe_call(robot.mechanical_joint_control, angle1=0, angle2=-15, angle3=-42, duration=1000)
    time.sleep(1.5)
    safe_call(robot.mechanical_clamp_release)
    time.sleep(1)
    print("[TASK 4] TOKEN PLACED IN BOX B — clamp released, token dropped")
    safe_call(robot.mecanum_move_speed_times, direction=1, speed=20, times=12, unit=1)
    safe_call(robot.mechanical_joint_control, angle1=0, angle2=40, angle3=40, duration=800)


# ---------------------------
# STAGE 1 — POSE CONTROL
# ---------------------------
def run_stage1(robot):
    print("[TASK 1] POSE RECOGNITION ACTIVE — waiting for pose input")

    # Open capture locally so it always works on re-run
    local_cap = cv2.VideoCapture(0)
    if not local_cap.isOpened():
        print("[ERROR] Cannot open webcam for pose recognition.")
        return

    history          = []
    _pose_recognised = False

    while True:
        ret, frame = local_cap.read()   # <-- use local_cap, not cap
        if not ret:
            break

        results = pose_model.predict(frame, verbose=False, imgsz=320, conf=0.35)
        person  = select_main_person(results)
        cmd = "NONE"
        if person is not None:
            kp_dict = {COCO_KEYPOINTS[i]: person[i] for i in range(len(COCO_KEYPOINTS))}
            cmd = classify_pose(kp_dict)
            draw_skeleton(frame, person)

        history.append(cmd)
        if len(history) > 6:
            history.pop(0)
        state = max(set(history), key=history.count)

        if state != "NONE" and not _pose_recognised:
            _pose_recognised = True
            print(f"[TASK 1] POSE RECOGNISED — command: {state}")

        robot_frame = get_robot_frame(robot)
        draw_ui(frame, state, robot_frame)
        cv2.imshow("UGOT CONTROL", frame)

        if robot is not None:
            if   state == "FORWARD":  safe_call(robot.mecanum_move_speed, 0, 20)
            elif state == "BACKWARD": safe_call(robot.mecanum_move_speed, 1, 20)
            elif state == "LEFT":     safe_call(robot.mecanum_turn_speed, 2, 30)
            elif state == "RIGHT":    safe_call(robot.mecanum_turn_speed, 3, 30)
            else:                     safe_call(robot.mecanum_stop)

        if state == "EXIT" or (cv2.waitKey(1) & 0xFF == 27):
            print("[TASK 2] MOVING TO BOX A — pose-controlled exit triggered")
            print("[TASK 2] STOPPED IN BOX A — on coloured mark")
            break

    local_cap.release()   # <-- release local, not global cap
    cv2.destroyAllWindows()
    if robot:
        safe_call(robot.mecanum_stop)
    time.sleep(0.5)


# ---------------------------
# STAGE 2 — FACE RECOGNITION (V3)
# ---------------------------
def run_stage2(robot, known_encs, known_names, target_input, spin_code):
    print(f"[TASK 3] SEARCHING FOR ASSIGNED FACE — target: {target_input}")

    confirmer         = FaceConfirmer(threshold=CONFIRM_THRESHOLD)
    is_mission_done   = False
    has_locked_target = False
    _face_recognised  = False

    frame_count    = 0
    cached_boxes   = []
    cached_labels  = {}

    # Zigzag search state (used only after losing a locked target)
    zigzag_step        = 0
    zigzag_frames_done = 0
    ZIGZAG_FRAMES      = 20

    # Show placeholder window immediately so waitKey works
    placeholder = np.zeros((480, 660, 3), dtype=np.uint8)
    cv2.putText(placeholder, f"Searching for {target_input}...",
                (60, 240), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
    cv2.imshow("Robot Tracking V3", placeholder)
    cv2.waitKey(1)

    while not is_mission_done:
        frame = get_robot_frame(robot)
        if frame is None:
            cv2.imshow("Robot Tracking V3", placeholder)
            if cv2.waitKey(50) & 0xFF == 27:
                break
            continue

        frame_count += 1
        orig_h, orig_w = frame.shape[:2]

        # Resize to working width
        scale     = PROCESS_WIDTH / orig_w
        small     = cv2.resize(frame, (PROCESS_WIDTH, int(orig_h * scale)))
        processed = preprocess_frame(small)
        rgb_small = cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)

        # Detection — only every SKIP_FRAMES frames
        if frame_count % SKIP_FRAMES == 0:
            locs, encs = detect_and_encode(rgb_small)

            new_boxes      = []
            new_labels     = {}
            active_indices = set()

            for idx, ((top, right, bottom, left), enc) in enumerate(zip(locs, encs)):
                active_indices.add(idx)

                l_orig = int(left   / scale)
                t_orig = int(top    / scale)
                r_orig = int(right  / scale)
                b_orig = int(bottom / scale)
                new_boxes.append((l_orig, t_orig, r_orig, b_orig))

                raw_name, dist_val = identify_face(enc, known_encs, known_names)
                stable_name = confirmer.update(idx, raw_name)
                new_labels[idx] = (stable_name, dist_val)

            confirmer.clear_stale(active_indices)
            cached_boxes  = new_boxes
            cached_labels = new_labels

        # Draw & find target
        found_target = False
        target_box   = None

        for idx, (l, t, r, b) in enumerate(cached_boxes):
            stable_name, dist_val = cached_labels.get(idx, ("UNKNOWN", 1.0))
            is_target = (stable_name == target_input)

            if is_target:
                found_target = True
                target_box   = (l, t, r, b)
                color        = (0, 255, 0)
            elif stable_name == "UNKNOWN":
                color = (100, 100, 100)
            else:
                color = (0, 80, 255)

            cv2.rectangle(frame, (l, t), (r, b), color, 2)
            label = f"{stable_name} ({dist_val:.2f})"
            cv2.putText(frame, label, (l, t - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

        # OCR fallback
        found_by_text = False
        try:
            res = robot.get_word_recognition_result() if robot else None
            if res:
                ocr_words = [str(w).upper() for w in res]
                found_by_text = any(target_input in w for w in ocr_words)
                cv2.putText(frame, f"OCR: {ocr_words}", (10, 55),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 0), 1)
        except Exception:
            pass

        # Movement logic
        if found_target or found_by_text:
            has_locked_target  = True
            zigzag_step        = 0
            zigzag_frames_done = 0
            safe_call(robot.mecanum_stop)

            if not _face_recognised:
                _face_recognised = True
                print(f"[TASK 3] ASSIGNED FACE RECOGNISED — {target_input} confirmed")

            if target_box:
                l, t, r, b = target_box
                cx = (l + r) // 2
                fw = r - l
            else:
                cx = orig_w // 2
                fw = 60

            err = cx - (orig_w // 2)

            if abs(err) > 40:
                direction = 2 if err < 0 else 3
                safe_call(robot.mecanum_turn_speed, direction, CENTER_SPEED)
                cv2.putText(frame, f"CENTERING err={err}", (10, 80),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)

            elif fw < 76.1:
                safe_call(robot.mecanum_move_speed, 0, APPROACH_FAST)

            else:
                print(f"[TASK 4] MOVING TO BOX B — target locked, face_w={fw}px")
                drop_apriltag(robot)
                is_mission_done = True

        else:
            if not has_locked_target:
                # Haven't found target yet — keep spinning to search
                safe_call(robot.mecanum_turn_speed, spin_code, SPIN_FAST)
            else:
                # Had target, briefly lost it — zigzag left/right to relocate
                zigzag_frames_done += 1
                if zigzag_frames_done >= ZIGZAG_FRAMES:
                    zigzag_frames_done = 0
                    zigzag_step += 1

                direction = 2 if (zigzag_step % 2 == 0) else 3
                safe_call(robot.mecanum_turn_speed, direction, SPIN_SLOW)

                cv2.putText(frame, f"ZIGZAG SEARCHING... step={zigzag_step}", (10, 80),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 165, 255), 1)

        # HUD
        hud = f"TARGET: {target_input} | {'FOUND ✓' if found_target else 'SEARCHING...'}"
        cv2.putText(frame, hud, (10, 28),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255) if found_target else (0, 165, 255), 2)

        cv2.imshow("Robot Tracking V3", frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break

    safe_call(robot.mecanum_stop)
    cv2.destroyAllWindows()


# ---------------------------
# MAIN
# ---------------------------
def run_system():
    if not known_names:
        print("❌ No faces in database — cannot continue.")
        return

    print(f"\nKnown faces: {known_names}")

    while True:
        target_input = input("Who should I find? ").strip().upper()
        if target_input in known_names:
            break
        print(f"  ⚠️  '{target_input}' not found. Choose from: {known_names}")

    while True:
        spin_dir = input("Spin direction to search (left / right): ").strip().lower()
        if spin_dir in ("left", "right"):
            break
        print("  ⚠️  Please type 'left' or 'right'.")

    spin_code = 2 if spin_dir == "left" else 3
    print(f"\n✅ Ready: find '{target_input}', spin {spin_dir}.")
    print("Starting Stage 1 (pose control)...\n")

    # Stage 1 — pose control
    run_stage1(robot)

    # Stage 2 — pass known_encs/known_names so no reload needed
    run_stage2(robot, known_encs, known_names, target_input, spin_code)

    print("\n[COMPLETE] Mission finished.")
    if robot:
        safe_call(robot.mecanum_stop)


if __name__ == "__main__":
    run_system()